In [39]:
import os
from typhoon_ocr import ocr_document
from bs4 import BeautifulSoup
import re
from rapidfuzz import fuzz
import pandas as pd

In [40]:
os.environ['TYPHOON_OCR_API_KEY'] = 'sk-ogvTcIwhoNXX69zmxalfZwZXW8JU5YcohnT4ISIgLTTapvJQ'

#### Extracion

In [41]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))
def extract_party_score_dict(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    table = soup.find("table")
    if table is None:
        raise ValueError("No <table> found")

    rows = table.find_all("tr")
    if not rows:
        return {}

    headers = [td.get_text(strip=True) for td in rows[0].find_all(["td", "th"])]

    # Candidate labels
    party_candidates = ["พรรคการเมือง", "ชื่อพรรคการเมือง", "สังกัดพรรคการเมือง"]
    score_candidates = ["ได้คะแนน"]

    party_idx = None
    score_idx = None
    best_party_score = 0
    best_score_score = 0

    for i, h in enumerate(headers):
        # compute best similarity across all candidates
        party_sim = max(fuzz.partial_ratio(h, c) for c in party_candidates)
        score_sim = max(fuzz.partial_ratio(h, c) for c in score_candidates)

        if party_sim > best_party_score:
            best_party_score = party_sim
            party_idx = i

        if score_sim > best_score_score:
            best_score_score = score_sim
            score_idx = i

    if best_party_score < 60 or best_score_score < 60:
        raise ValueError("Required columns not confidently found")

    result = {}

    for row in rows[1:]:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]

        if len(cols) <= max(party_idx, score_idx):
            continue

        key = cols[party_idx]
        value = cols[score_idx]

        try:
            value = thai_num_to_int(value)
        except Exception:
            continue  # safer: skip invalid rows

        result[key] = value

    return result
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        markdown = ocr_document(
            pdf_or_image_path=path
        )
        party_dict = extract_party_score_dict(markdown)
        return party_dict
    except Exception as e:
        print(e.__str__())
        return {}

In [42]:
template_df = pd.read_csv("../data/submission_template.csv")
template_df

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0
...,...,...,...,...,...
10048,party_list_34_11_53,party_list_34_11,53,ไทยพิทักษ์ธรรม,0
10049,party_list_34_11_54,party_list_34_11,54,ความหวังใหม่,0
10050,party_list_34_11_55,party_list_34_11,55,ไทยรวมไทย,0
10051,party_list_34_11_56,party_list_34_11,56,เพื่อบ้านเมือง,0


In [43]:
old_submission_df = pd.read_csv("./submission.csv")
old_submission_df

,id,votes
0,constituency_10_1_1,14813
1,constituency_10_1_2,14368
2,constituency_10_1_3,979
3,constituency_10_1_4,244
4,constituency_10_1_5,351
...,...,...
10048,party_list_34_11_53,14
10049,party_list_34_11_54,41
10050,party_list_34_11_55,0
10051,party_list_34_11_56,29


In [44]:
old_submission_df['doc_id'] = template_df['doc_id']
old_submission_df['row_num'] = template_df['row_num']
old_submission_df['party_name'] = template_df['party_name']
old_submission_df

,id,votes,doc_id,row_num,party_name
0,constituency_10_1_1,14813,constituency_10_1,1,ประชาธิปัตย์
1,constituency_10_1_2,14368,constituency_10_1,2,ภูมิใจไทย
2,constituency_10_1_3,979,constituency_10_1,3,เศรษฐกิจ
3,constituency_10_1_4,244,constituency_10_1,4,กล้าธรรม
4,constituency_10_1_5,351,constituency_10_1,5,พลวัต
...,...,...,...,...,...
10048,party_list_34_11_53,14,party_list_34_11,53,ไทยพิทักษ์ธรรม
10049,party_list_34_11_54,41,party_list_34_11,54,ความหวังใหม่
10050,party_list_34_11_55,0,party_list_34_11,55,ไทยรวมไทย
10051,party_list_34_11_56,29,party_list_34_11,56,เพื่อบ้านเมือง


##### TIME TO FIX

In [45]:
old_submission_df[old_submission_df['votes'] == 0]

,id,votes,doc_id,row_num,party_name
33,constituency_10_10_16,0,constituency_10_10,16,วิชชั่นใหม่
61,constituency_10_12_10,0,constituency_10_12,10,ไทยก้าวใหม่
68,constituency_10_13_1,0,constituency_10_13,1,ประชาชน
74,constituency_10_13_7,0,constituency_10_13,7,รักชาติ
78,constituency_10_13_11,0,constituency_10_13,11,โอกาสใหม่
...,...,...,...,...,...
9994,party_list_34_10_56,0,party_list_34_10,56,เพื่อบ้านใหม่
9995,party_list_34_10_57,0,party_list_34_10,57,พลังไทยรักชาติ
10015,party_list_34_11_20,0,party_list_34_11,20,ฟิวชัน
10031,party_list_34_11_36,0,party_list_34_11,36,ไทยพร้อม


In [46]:
print(len(old_submission_df[old_submission_df['votes'] == 0]['doc_id'].unique()))
for file in old_submission_df[old_submission_df['votes'] == 0]['doc_id'].unique():
    print(file)

192
constituency_10_10
constituency_10_12
constituency_10_13
constituency_10_14
constituency_10_16
constituency_10_17
constituency_10_20
constituency_10_21
constituency_10_23
constituency_10_25
constituency_10_26
constituency_10_3
constituency_10_31
constituency_10_33
constituency_10_5
constituency_10_6
constituency_12_2
constituency_12_4
constituency_13_2
constituency_13_4
constituency_13_5
constituency_14_2
constituency_16_4
constituency_19_2
constituency_19_4
constituency_20_1
constituency_20_10
constituency_20_6
constituency_22_2
constituency_24_1
constituency_26_2
constituency_27_2
constituency_30_1
constituency_30_10
constituency_30_13
constituency_30_14
constituency_30_15
constituency_30_5
constituency_31_4
constituency_32_1
constituency_33_1
constituency_33_3
constituency_33_4
constituency_33_9
party_list_10_1
party_list_10_10
party_list_10_11
party_list_10_12
party_list_10_13
party_list_10_14
party_list_10_16
party_list_10_17
party_list_10_18
party_list_10_19
party_list_10_2
p

##### FIXING

In [56]:
vote = extraction("../constituency_10_1_page2_crop.png")
vote

{'ประชาชน': 34167,
 'ประชาธิปัตย์': 14813,
 'ภูมิใจไทย': 14368,
 'เพื่อไทย': 6030,
 'รวมไทยสร้างชาติ': 2075,
 'โอกาสใหม่': 1133,
 'ไทยภักดี': 1023,
 'เศรษฐกิจ': 979,
 'ไทยสร้างไทย': 629,
 'ไทยก้าวใหม่': 489,
 'พลวัต': 351,
 'กล้าธรรม': 244,
 'ปวงชนไทย': 168,
 'รักชาติ': 165,
 'ทางเลือกใหม่': 154,
 'วิชชั่นใหม่': 113,
 'ประชาธิปไตยใหม่': 94,
 'พลังประชารัฐ': 80}

In [47]:
PREFIX = "../data/images/"

In [48]:
now = "constituency_10_10"

In [55]:
vote = extraction(PREFIX + now + '_page2.png')

In [54]:
vote

{}

In [50]:
old_submission_df[old_submission_df['doc_id'] == now]

,id,votes,doc_id,row_num,party_name
18,constituency_10_10_1,41804,constituency_10_10,1,ประชาชน
19,constituency_10_10_2,19047,constituency_10_10,2,เพื่อไทย
20,constituency_10_10_3,9440,constituency_10_10,3,โอกาสใหม่
21,constituency_10_10_4,7925,constituency_10_10,4,ภูมิใจไทย
22,constituency_10_10_5,6372,constituency_10_10,5,ประชาธิปัตย์
23,constituency_10_10_6,2012,constituency_10_10,6,รวมไทยสร้างชาติ
24,constituency_10_10_7,1437,constituency_10_10,7,เศรษฐกิจ
25,constituency_10_10_8,636,constituency_10_10,8,ไทยสร้างไทย
26,constituency_10_10_9,583,constituency_10_10,9,ไทยก้าวใหม่
27,constituency_10_10_10,545,constituency_10_10,10,กล้าธรรม
